<a href="https://colab.research.google.com/github/Arn099xx/StormDrainBlockageDetector/blob/main/TransferModel(SDBD).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
from tensorflow import keras
from keras import models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.applications.mobilenet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
#PREPROCESS DATA

IMG_SIZE = (180,180)
BATCH_SIZE = 32
TRAINABLE_LAYERS = 0 #PLACEHOLDER

train_dir = 'TBD'
val_dir = 'TBD'
test_dir = 'TBD'

#Augmentation AND Preprocessing for MobileNet
train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input,
                                   rotation_range = 20,
                                   width_shift_range = 0.2,
                                   height_shift_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True,
                                   vertical_flip=True)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

validation_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)




In [ ]:
#Load MobileNet

base_model = MobileNet(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)

#Freezing layers
for layer in base_model.layers:
  layer.trainable=False

#Unfreeze last n layers

if TRAINABLE_LAYERS > 0:
  for layer in base_model.layers[-TRAINABLE_LAYERS:]:
    layer.trainable=True

#TRANSFER MODEL

transfer_model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dense(train_generator.num_classes, activation='softmax') #? Should we define 3 classes, or leave it flexible?
])




In [ ]:
#Compiling the Model

transfer_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
#Train

transfer_model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator
)